# 因子数据模型

因子数据模型采用了三层结构:
* 最上层是因子库: `QuantStudio.Factor.FactorDB.FactorDB`
* 因子库包含多张因子表: `QuantStudio.Factor.FactorTable.FactorTable`
* 每张因子表中又包含多个因子: `QuantStudio.Factor.Factor.Factor`

每张因子表的数据逻辑上是一个三维数组, 第一维是因子名称, 第二维是时间点, 第三维是证券代码. 具体到程序里的数据类型, 以 Panel(`QuantStudio.Core.QSObject.Panel`)数据类型组织. QuantStudio 规定了这三个维度的先后顺序, 对应于 Panel 数据类型, items 是因子, major_axis 是时点, minor_axis 是证券代码. 在 QuantStudio 的所有 API 中, 凡是涉及到因子数据的地方, 都将遵守此组织原则。

每个因子的数据逻辑上是一个 DataFrame, index 是时间点, columns 是证券代码。

另外, 对于时间点, 采用 Python 的 datetime 表示, 证券代码的数据类型为字符串, 例如, “000001.SZ”, 因子名称的数据类型也是字符串。

In [2]:
# 因子数据模型
FactorFramework = """
graph TB
    subgraph FactorFramework["因子框架"]
        direction TB

        FactorDB["因子库"]

        FactorTable1["因子表1"]
        FactorTable2["因子表2"]

        subgraph FT1["因子表1内容"]
            F11["因子1"]
            F12["因子2"]
            F1M["因子M"]
        end

        subgraph FT2["因子表2内容"]
            F21["因子1"]
            F22["因子2"]
            F2N["因子N"]
        end
        
    end
    
    FactorDB --> FactorTable1
    FactorDB --> FactorTable2
    
    FactorTable1 --> F11
    FactorTable1 --> F12
    FactorTable1 --> F1M
    
    FactorTable2 --> F21
    FactorTable2 --> F22
    FactorTable2 --> F2N
    
    DF["DataFrame<br/>- index: 时间<br/>- columns: 证券代码"]
    
    F11 --> DF
    F12 --> DF
    F1M --> DF
    F21 --> DF
    F22 --> DF
    F2N --> DF

    style FactorFramework fill:#e1f5fe
    style FactorDB fill:#f87f89
    style FactorTable1 fill:#efbaf7
    style FactorTable2 fill:#efbaf7
    style DF fill:#fff3e0
"""

from mermaid import Mermaid
display(Mermaid(FactorFramework))

# Panel 数据对象

因子表或者多个因子的数据以 Panel([`QuantStudio.Core.QSObject.Panel`](https://github.com/Scorpi000/QuantStudio/blob/d67413f0e8bb26391ddd1c2a6b67c28e66bbac59/QuantStudio/Core/QSObject.py#L870))数据类型组织，这个是 pandas 早期版本中 Panel 对象的复刻，但只保留了必要的功能，主要是为了表达数据。

In [18]:
from QuantStudio.Core.QSObject import Panel
from QuantStudio.Tools.Visualization import qs_help

print(qs_help(Panel))

类型: class
模块: QuantStudio.Core.QSObject
构造函数签名: Panel.__init__(self, data=None, items: Union[ForwardRef('ExtensionArray'), numpy.ndarray, ForwardRef('Index'), ForwardRef('Series'), pandas._typing.SequenceNotStr, range, NoneType] = None, major_axis: Union[ForwardRef('ExtensionArray'), numpy.ndarray, ForwardRef('Index'), ForwardRef('Series'), pandas._typing.SequenceNotStr, range, NoneType] = None, minor_axis: Union[ForwardRef('ExtensionArray'), numpy.ndarray, ForwardRef('Index'), ForwardRef('Series'), pandas._typing.SequenceNotStr, range, NoneType] = None)
构造函数文档:
    Args:
        data: 输入数据, 可接受的类型:
            * Dict of 2D numpy.ndarrays, Iterable, or DataFrame
            * 3-D numpy.ndarray, Iterable
        items: Index or array-like, 用于第一个维度的索引。如果输入数据 data 中不包含索引信息，且未显式提供索引，则默认为 RangeIndex。
        major_axis: Index or array-like, 用于第二个维度的索引。如果输入数据 data 中不包含索引信息，且未显式提供索引，则默认为 RangeIndex。
        minor_axis: Index or array-like, 用于第三个维度的索引。如果输入数据 data 中不包含索引信息，且未显式提供索引，则默认为 RangeIn

In [19]:
# 创建 Panel
import datetime as dt

import numpy as np
import pandas as pd
np.random.seed(0)

DTs = pd.date_range(dt.datetime(2025, 1, 1), dt.datetime(2025, 1, 4))
IDs = ["000001.SZ", "000002.SZ", "000003.SZ"]
p = Panel(data=np.random.rand(2, 4, 3), items=["close", "open"], major_axis=DTs, minor_axis=IDs)

print(p)

<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 2 (items) x 4 (major_axis) x 3 (minor_axis)
Items axis: close to open
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-04 00:00:00
Minor_axis axis: 000001.SZ to 000003.SZ


In [20]:
# 通过位置索引数据
print(p.iloc[0, 0:3, [0, 2]])

            000001.SZ  000003.SZ
2025-01-01   0.548814   0.602763
2025-01-02   0.544883   0.645894
2025-01-03   0.437587   0.963663


In [21]:
# 通过标签索引数据
print(p.loc["close", dt.datetime(2025, 1, 1):dt.datetime(2025, 1, 3), ["000001.SZ", "000003.SZ"]])

            000001.SZ  000003.SZ
2025-01-01   0.548814   0.602763
2025-01-02   0.544883   0.645894
2025-01-03   0.437587   0.963663


In [22]:
# Panel 转 MultiIndex DataFrame
print(p.to_frame(filter_observations=False))

                         close      open
2025-01-01 000001.SZ  0.548814  0.568045
           000002.SZ  0.715189  0.925597
           000003.SZ  0.602763  0.071036
2025-01-02 000001.SZ  0.544883  0.087129
           000002.SZ  0.423655  0.020218
           000003.SZ  0.645894  0.832620
2025-01-03 000001.SZ  0.437587  0.778157
           000002.SZ  0.891773  0.870012
           000003.SZ  0.963663  0.978618
2025-01-04 000001.SZ  0.383442  0.799159
           000002.SZ  0.791725  0.461479
           000003.SZ  0.528895  0.780529


In [24]:
# Panel 转 numpy.ndarray
print(p.values)

[[[0.5488135  0.71518937 0.60276338]
  [0.54488318 0.4236548  0.64589411]
  [0.43758721 0.891773   0.96366276]
  [0.38344152 0.79172504 0.52889492]]

 [[0.56804456 0.92559664 0.07103606]
  [0.0871293  0.0202184  0.83261985]
  [0.77815675 0.87001215 0.97861834]
  [0.79915856 0.46147936 0.78052918]]]
